# Pympact development
## How many asteroids are out there?

In [24]:
import multineas as neas
import multineas.legacyorb as leg
import numpy as np
import spiceypy as spy
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Compute numerical Jacobians

Let's test the numerical Jacobian with a simple transformation:

$$
x = r \cos\theta\\
y = r \sin\theta
$$

we have two sets of coordinates: $X :(x,y)$ and $E: (r,\theta)$

The Jacobian matrix for going from $E$ to $X$, ie. $dX = J_{XoE} dE$ is:

$$
J_{XoE} = \left(
\begin{array}{cc}
\partial x/\partial r & \partial x/\partial \theta \\
\partial y/\partial r & \partial y/\partial \theta \\
\end{array}
\right)
=
\left(
\begin{array}{cc}
\cos\theta & -r\sin\theta \\
\sin\theta & r\cos\theta \\
\end{array}
\right)
$$

The reason why I call $J_{XoE}$, that must be read "$X$ over $E$" is because:

$$
J_{XoE} = \frac{dx dy}{dr d\theta} = \frac{dX}{dE}
$$

In [8]:
#Routine going from E to X
def EtoX(E):
    r,q=E
    x=r*np.cos(q)
    y=r*np.sin(q)
    X=np.array([x,y])
    return X

# Value of E
r=2
q=np.pi/3
E=[r,q]
dE=[1e-3, 1e-3]

# Compute numerical
X, JXoE=leg.computeNumericalJacobian(EtoX,E,dE)

# Verify
X, JXoE, np.array([[np.cos(q),-r*np.sin(q)],[np.sin(q),r*np.cos(q)]])

(array([1.        , 1.73205081]),
 array([[ 0.5       , -1.73205052],
        [ 0.8660254 ,  0.99999983]]),
 array([[ 0.5       , -1.73205081],
        [ 0.8660254 ,  1.        ]]))

Now the inverse:

In [13]:
# From cartesian to sperical
def XtoE(X):
    x,y=X
    r=(x**2+y**2)**0.5
    q=np.arctan2(y,x)
    E=np.array([r,q])
    return E

# Value
X=np.array([1.0,1.73205081])
dX=[1e-3, 1e-3]

# Jacobian
E, JEoX = leg.computeNumericalJacobian(XtoE, X, dX)
E, JEoX

(array([2.        , 1.04719755]),
 array([[ 0.49999995,  0.86602538],
        [-0.4330127 ,  0.25000004]]))

We can verify that $J_{XoE}$ is the inverse of $J_{EoX}$:

In [14]:
JXoE, np.linalg.inv(JEoX)

(array([[ 0.5       , -1.73205052],
        [ 0.8660254 ,  0.99999983]]),
 array([[ 0.50000009, -1.73205076],
        [ 0.86602541,  0.99999991]]))

And the difference that should be equal to zero:

In [15]:
np.linalg.norm(JEoX-np.linalg.inv(JXoE))

np.float64(9.063948197339815e-08)

Now we want to predict the number of points falling in a rectangle $[x_{cen}-dx, x_{cen}+dx]$ and $[y_{cen}-dy, y_{cen}+dy]$ after generating uniformly $r$ and $\theta$

Let's do it numerically:

In [19]:
#np.random.seed(1)
N = 1000000
rs = np.random.uniform(low=0, high=1, size=N)
thetas = np.random.uniform(low=0, high=2*np.pi, size=N)
xs = rs*np.cos(thetas)
ys = rs*np.sin(thetas)

xcen = 0.5; dx = 0.1
ycen = 0.5; dy = 0.1
cond = (xs<=xcen+dx)*(xs>=xcen-dx)*(ys<=ycen+dy)*(ys>=ycen-dy)
xsel = xs[cond]
ysel = ys[cond]
Nnum = cond.sum()
Nnum

np.int64(9077)

Theoretically the number can be computed using:

$$
p(X)|dX| = p(E) |dE|
$$
then

$$
p(X) = \left|\frac{dE}{dX}\right| p(E)
$$

There we identify the determinant of the Jacobian:

$$
p(X) = \det(J_{EoX})\;p(E)
$$

The number of points will be:

$$
N_{teo} \approx N p(X) (2dx)(2dy) = 4 N \det(J_{EoX})\;p(E) dx dy
$$

In [20]:
r = (xcen**2 + ycen**2)**0.5
q = np.atan2(ycen, xcen)
JXoE = np.array([[np.cos(q),-r*np.sin(q)],[np.sin(q),r*np.cos(q)]])
JEoX = np.linalg.inv(JXoE)

pE = 1/(1-0)*1/(2*np.pi-0)
Nteo = N*np.linalg.det(JEoX)*pE*(2*dx)*(2*dy)

Nteo

np.float64(9003.163161571061)

Which almost coincides with $N_{num}$

In [21]:
Nnum

np.int64(9077)

## Orbital elements

Let's do it with orbital elements:

In [25]:
def X2E(X,mu):
    elts=spy.oscelt(X,0,mu)
    E=elts[:6]
    return E

mu=1
X=np.array([1,1,1,-0.1,-0.1,1])
dX=np.array([1e-3]*6)
args=dict(mu=mu)
E,JEoX=leg.computeNumericalJacobian(X2E,X,dX,**args)

E, JEoX

(array([1.32894739, 0.82099007, 1.57079633, 0.78539816, 5.83284997,
        0.07211871]),
 array([[ 0.92584986,  0.92584986, -0.14602651, -1.12472527, -1.12472527,
          0.52851048],
        [ 0.38679954,  0.38679954,  0.53118186, -0.11428948, -0.11428948,
          2.58670157],
        [ 0.06428244, -0.06428244,  0.        ,  0.64282432, -0.64282432,
          0.        ],
        [-0.45454552,  0.45454552,  0.        ,  0.45454552, -0.45454552,
          0.        ],
        [ 0.62344381,  0.62344381, -0.18087844, -1.69081212, -1.69081212,
          1.79386759],
        [-0.31924886, -0.31924886, -0.26374428,  0.23132464,  0.23132464,
         -1.75821816]]))

This is the analytical formulae:

In [26]:
def calcKeplerianJacobians(mu,celements,state):
    """
    Compute the Jacobian Matrix of the transformation from classical
    orbital elements (q,e,i,w,W,M) to cartesian state vector (x,y,z,x',y',z').

    Parameters:
        mu: Gravitational parameter.
        celements: Cometary elements (q,e,i,w,W,M)

    Return:

        det JXoc, where JXoc = JXoe * Jeoc, and:

            JXoe = [dx/da,dx/de,dx/di,dx/dw,dx/dW,dx/dM,
                    dy/da,dy/de,dy/di,dy/dw,dy/dW,dy/dM,
                    dz/da,dz/de,dz/di,dz/dw,dz/dW,dz/dM,
                    dx'/da,dx'/de,dx'/di,dx'/dw,dx'/dW,dx'/dM,
                    dy'/da,dy'/de',dy'/di,dy'/dw,dy'/dW,dy'/dM,
                    dz'/da,dz'/de,dz'/di',dz'/dw,dz'/dW,dz'/dM],

                    Numpy array 6x6, units compatible with mu and a.

        and:

            Jeoc = [da/dq,da/de,...]

            Jeoc = [1/(1-e),q/(1-e)**2,0,0,0,0,
                    0      ,1         ,0,0,0,0,
                    0      ,0         ,1,0,0,0,
                    0      ,0         ,0,1,0,0,
                    0      ,0         ,0,0,1,0,
                    0      ,0         ,0,0,0,1,
                    ]

            det Jeoc = 1/(1-e)

        Jacobians are used for transform pdf:

                p(c) = p(X) det(JXoc)
    """
    q,e,i,W,w,M=celements
    a=q/(1-e)

    #Orbit signature
    if e<1:
        s=+1
    elif e>1:
        s=-1
    else:
        s=0

    #Trigonometric function
    cosi,sini=np.cos(i), np.sin(i)
    cosw,sinw=np.cos(w), np.sin(w)
    cosW,sinW=np.cos(W), np.sin(W)

    #Components of the rotation matrix
    A=(cosW*cosw-cosi*sinW*sinw);B=(-cosW*sinw-cosw*cosi*sinW)
    C=(cosw*sinW+sinw*cosi*cosW);D=(-sinw*sinW+cosw*cosi*cosW)
    F=sinw*sini;G=cosw*sini

    #Primary auxiliar variables
    ab=np.abs(a)
    n=np.sqrt(mu/ab**3)
    nu=n*a**2
    eps=np.sqrt(s*(1-e**2))

    #Get cartesian coordinates
    x,y,z,vx,vy,vz=state
    r=(x**2+y**2+z**2)**0.5
    nur=nu/r

    #Eccentric anomaly as obtained from indirect information
    #From the radial equation: r = a (1-e cos E)
    cosE=(1/e)*(1-r/a)

    #From the general equation for y
    #NOTE: This is the safest way to obtain sinE without the danger of singularities
    sinE=(y-a*(cosE-e)*C)/(ab*eps*D)

    #dX/da
    Ja=np.array([x/a,y/a,z/a,-vx/(2*a),-vy/(2*a),-vz/(2*a)])

    #dX/de
    dcosEde=-s*a*sinE**2/r
    dsinEde=a*cosE*sinE/r
    dnurde=(nu*a/r**2)*(cosE-(ab/r)*e*sinE**2)
    depsde=-s*e/eps

    drAde=a*(dcosEde-1)
    drBde=ab*(depsde*sinE+eps*dsinEde)

    dvAde=-(dnurde*sinE+nur*dsinEde)
    dvBde=(dnurde*eps*cosE+nur*depsde*cosE+nur*eps*dcosEde)

    Je=np.array([
        drAde*A+drBde*B,
        drAde*C+drBde*D,
        drAde*F+drBde*G,
        dvAde*A+dvBde*B,
        dvAde*C+dvBde*D,
        dvAde*F+dvBde*G,
    ])

    #dX/di
    Ji=np.array([z*sinW,-z*cosW,-x*sinW+y*cosW,vz*sinW,-vz*cosW,-vx*sinW+vy*cosW])

    #dX/dw
    Jw=np.array([-y*cosi-z*sini*cosW,x*cosi-z*sini*sinW,sini*(x*cosW+y*sinW),
                    -vy*cosi-vz*sini*cosW,vx*cosi-vz*sini*sinW,sini*(vx*cosW+vy*sinW)])

    #dX/dW
    JW=np.array([-y,x,0,-vy,vx,0])

    #dX/dM
    JM=np.concatenate(((ab**3/mu)**0.5*np.array([vx,vy,vz]),
                        (mu*ab**3)**0.5*np.array([-x/r**3,-y/r**3,-z/r**3])))

    #Jacobian
    JX2e=np.array([Ja,Je,Ji,JW,Jw,JM]).transpose()

    #Jacobian from classical elements (a) to cometary elements (q)
    Je2c=np.eye(6)
    Je2c[0,0]=1/(1-e)
    Je2c[0,1]=q/(1-e)**2
    JX2c=np.matmul(JX2e,Je2c)

    return JX2c

In [27]:
X = spy.conics(list(E)+[0, mu], 0)
JXoE = calcKeplerianJacobians(mu,E,X)
JXoE

array([[ 7.52475236e-01, -1.03827530e+00,  7.07106781e-01,
        -1.00000000e+00, -7.07106781e-01, -2.02276767e+00],
       [ 7.52475236e-01, -1.03827530e+00, -7.07106781e-01,
         1.00000000e+00, -7.07106781e-01, -2.02276767e+00],
       [ 7.52475236e-01,  1.26145562e+01,  1.11022302e-16,
         0.00000000e+00,  1.41421356e+00,  2.02276767e+01],
       [ 3.76237618e-02, -2.16331960e+00,  7.07106781e-01,
         1.00000000e-01, -7.07106781e-01, -3.89281819e+00],
       [ 3.76237618e-02, -2.16331960e+00, -7.07106781e-01,
        -1.00000000e-01, -7.07106781e-01, -3.89281819e+00],
       [-3.76237618e-01, -2.08446642e+00,  5.55111512e-17,
         0.00000000e+00, -1.41421356e-01, -3.89281819e+00]])

We can verify that the matrices are correct:

In [28]:
JXoE - np.linalg.inv(JEoX)

array([[ 1.74898911e-06, -1.33468751e-05, -1.55842153e-08,
        -1.50137150e-07, -6.02027869e-06, -2.16159327e-05],
       [ 1.74898911e-06, -1.33468751e-05,  1.55839155e-08,
         1.50137574e-07, -6.02027869e-06, -2.16159327e-05],
       [-2.69981262e-06,  2.37331139e-04,  3.75794766e-13,
        -5.31757799e-13,  6.96087439e-05,  4.04141055e-04],
       [ 2.40512929e-06, -2.95660839e-05, -1.55841448e-08,
         1.37786449e-09, -1.10791716e-05, -4.84124263e-05],
       [ 2.40512929e-06, -2.95660839e-05,  1.55839844e-08,
        -1.37763766e-09, -1.10791716e-05, -4.84124263e-05],
       [ 5.97575308e-07, -4.29397776e-05, -8.99784532e-15,
         1.28573391e-14, -1.25733512e-05, -7.26089135e-05]])